In [2]:
# codigo inicial que unificou as bases de vendas e vistoria, criou as colunas de ágio absoluto e percentual, 
# e removeu os registros duplicados dos itens agrupados
# removeu os imoveis da tipologia apartamento/nao é caracteristica da empresa. Venda por convenio de outra instituicao
# teste usando o codigo da destinacao original, via onehot encoding

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error, r2_score, mean_absolute_percentage_error,
    root_mean_squared_error
)

import matplotlib.pyplot as plt

tabela = pd.read_csv("base_vendas_atividade2_final1.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("base_vistorias_atividade2_final.csv", sep=";", encoding="latin-1")
destinacoes = pd.read_csv("base_destinacoes_atividade2_final.csv", sep=";", encoding="latin-1")

tabela = tabela.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)


tabela = tabela.merge(
    destinacoes[["COD_DESTINACAO_IMOVEL", "TIPO_USO_DESTINACAO", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]],
    how="left",
    left_on="COD_DESTINACAO_IMOVEL",
    right_on="COD_DESTINACAO_IMOVEL"
)       


# # remove a coluna duplicada da chave
tabela = tabela.drop(columns=["CD_IMOVEL_URBANO", "CD_IMOVEL"])
tabela = tabela.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
# tabela["SITUACAO_VISTORIA"] = tabela["SITUACAO_VISTORIA"].fillna("SEM_VISTORIA")


# # colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# # máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = tabela.duplicated(subset=cols, keep=False)
# # remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = tabela.loc[~mask_dup].copy()

# # Foram removidos 145 itens que constavam como items agrupados, de um total de 2982, restando 2837 registros 
print("Linhas originais:", len(tabela))
print("Linhas removidas:", mask_dup.sum())
print("Linhas finais:", len(tabela_sem_dups))

tabela = tabela_sem_dups
tabela["NR_EDITAL"] = tabela["ANO_VENDA"].astype(str) + "-" + tabela["NR_EDITAL"].astype(str)

def br_to_float(s):
    """
    Converte número no formato BR para float:
    - remove separador de milhar (.)
    - troca decimal (,) por (.)
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "":
        return np.nan
    s = s.replace(".", "")      # remove milhares
    s = s.replace(",", ".")     # troca decimal
    return pd.to_numeric(s, errors="coerce")

cols = ["VALOR_VENDA", "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "VALOR_LAUDO"]  # ajuste
for c in cols:
    if c in tabela.columns:
        tabela[c] = tabela[c].apply(br_to_float)

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
tabela["AGIO_ABSOLUTO"] = tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]
tabela["AGIO_PERCENTUAL"] = ((tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]) / tabela["VALOR_LAUDO"]) * 100

#retirando os apartamentos da base (14 registros) que tem área máxima de construção igual a zero e área base igual a zero, ou seja, não tem área construída, o que é um erro de cadastro
tabela = tabela[~((tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0))]   






Linhas originais: 2982
Linhas removidas: 145
Linhas finais: 2837


In [3]:
# -------------------------
# 0) DADOS (ajuste se necessário)
# -------------------------
df = tabela.copy()
df.columns = df.columns.str.strip()

TARGET = "VALOR_VENDA"


# Remover colunas que você NÃO quer usar
# (vazamento/colinearidade/dado futuro)
DROP_COLS = ["AGIO_ABSOLUTO", "AGIO_PERCENTUAL", "VALOR_LAUDO", "QTD_OFERTAS", "DS_DESTINACAO_IMOVEL"]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# Percentual 0–100 -> 0–1
if "PERCENTUAL_ENTRADA" in df.columns:
    df["PERCENTUAL_ENTRADA"] = pd.to_numeric(df["PERCENTUAL_ENTRADA"], errors="coerce") / 100.0


# Separar X e y
y = df[TARGET].copy()
X = df.drop(columns=[TARGET] + DROP_COLS)

# DS_CIDADE;DS_SETOR;COD_DESTINACAO_IMOVEL;TIPO_USO_DESTINACAO;
# Definir colunas numéricas e categóricas explicitamente
num_cols = [
    "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "PERCENTUAL_ENTRADA", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]

num_cols = [c for c in num_cols if c in X.columns]

cat_cols = ["DS_CIDADE", "DS_SETOR", "SITUACAO_VISTORIA", "COD_DESTINACAO_IMOVEL", "ANO_VENDA", "NR_EDITAL", "TIPO_USO_DESTINACAO", "ITEM_EDITAL"]
cat_cols = [c for c in cat_cols if c in X.columns]

# Tipos (evita problemas de mixed types)
for c in cat_cols:
    X[c] = X[c].astype("string")
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)
print("Shape X:", X.shape, "| Shape y:", y.shape)

Numéricas: ['AREA_MAX_CONSTR', 'AREA_BASE', 'AREA', 'PERCENTUAL_ENTRADA', 'RESIDENCIAL', 'COMERCIAL', 'INDUSTRIAL', 'INSTITUCIONAL']
Categóricas: ['DS_CIDADE', 'DS_SETOR', 'SITUACAO_VISTORIA', 'COD_DESTINACAO_IMOVEL', 'ANO_VENDA', 'NR_EDITAL', 'TIPO_USO_DESTINACAO', 'ITEM_EDITAL']
Shape X: (2823, 16) | Shape y: (2823,)


In [4]:
# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Transformação log do preço (recomendado pela cauda longa)
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)


# ============================================================
# 1) PREPROCESSAMENTO
# ============================================================
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe_sparse = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess_sparse = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe_sparse, cat_cols),
    ],
    remainder="drop"
)

# Dense (necessário para KNN e SVR; e para HGB no seu caso)
cat_pipe_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess_dense = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe_dense, cat_cols),
    ],
    remainder="drop"
)


In [5]:
# ============================================================
# 2) FUNÇÕES AUXILIARES: TREINO + TESTE
# ============================================================
def _metrics_rs(y_true, y_pred):
    return {
        "MAE_R$": mean_absolute_error(y_true, y_pred),
        "RMSE_R$": root_mean_squared_error(y_true, y_pred),
        "R2_R$": r2_score(y_true, y_pred),
        "MAPE_R$": mean_absolute_percentage_error(y_true, y_pred),
    }

def _metrics_log(y_true_log, y_pred_log):
    return {
        "MAE_log": mean_absolute_error(y_true_log, y_pred_log),
        "RMSE_log": root_mean_squared_error(y_true_log, y_pred_log),
    }

def eval_train_test(best_estimator, model_name, piso_mape=100_000):
    """
    Avalia no TREINO e no TESTE.
    Lembrete: o modelo foi treinado no LOG (y_train_log).
    """
    # ---- TREINO
    pred_log_train = best_estimator.predict(X_train)
    pred_train = np.expm1(pred_log_train)

    m_train = _metrics_rs(y_train, pred_train)
    m_train_log = _metrics_log(y_train_log, pred_log_train)

    # MAPE com piso no treino
    mask_tr = y_train >= piso_mape
    m_train[f"MAPE_R$_>={piso_mape}"] = (
        mean_absolute_percentage_error(y_train[mask_tr], pred_train[mask_tr]) if mask_tr.any() else np.nan
    )

    # ---- TESTE
    pred_log_test = best_estimator.predict(X_test)
    pred_test = np.expm1(pred_log_test)

    m_test = _metrics_rs(y_test, pred_test)
    m_test_log = _metrics_log(y_test_log, pred_log_test)

    mask_te = y_test >= piso_mape
    m_test[f"MAPE_R$_>={piso_mape}"] = (
        mean_absolute_percentage_error(y_test[mask_te], pred_test[mask_te]) if mask_te.any() else np.nan
    )

    print(f"\n=== {model_name} (TREINO) ===")
    for k, v in {**m_train, **m_train_log}.items():
        print(f"{k}: {v}")

    print(f"\n=== {model_name} (TESTE) ===")
    for k, v in {**m_test, **m_test_log}.items():
        print(f"{k}: {v}")

    # retorno em uma linha (para tabela)
    return {
        "Modelo": model_name,
        # treino
        "MAE_R$_treino": m_train["MAE_R$"],
        "RMSE_R$_treino": m_train["RMSE_R$"],
        "R2_R$_treino": m_train["R2_R$"],
        "MAPE_R$_treino": m_train["MAPE_R$"],
        f"MAPE_R$_>={piso_mape}_treino": m_train[f"MAPE_R$_>={piso_mape}"],
        "RMSE_log_treino": m_train_log["RMSE_log"],
        # teste
        "MAE_R$_teste": m_test["MAE_R$"],
        "RMSE_R$_teste": m_test["RMSE_R$"],
        "R2_R$_teste": m_test["R2_R$"],
        "MAPE_R$_teste": m_test["MAPE_R$"],
        f"MAPE_R$_>={piso_mape}_teste": m_test[f"MAPE_R$_>={piso_mape}"],
        "RMSE_log_teste": m_test_log["RMSE_log"],
    }

def gridsearch(name, pipe, param_grid, cv=5):
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv,
        scoring="neg_root_mean_squared_error",  # RMSE no LOG
        n_jobs=-1,
        verbose=1,
        error_score="raise"
    )
    gs.fit(X_train, y_train_log)
    print(f"\n{name} best params:", gs.best_params_)
    print(f"{name} best CV RMSE(log):", -gs.best_score_)
    return gs

In [6]:
# ============================================================
# 3) RIDGE
# ============================================================
pipe_ridge = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", Ridge(random_state=42))
])

param_grid_ridge = {
    "model__alpha": [0.1, 1, 3, 10, 30, 100, 300],
    "model__fit_intercept": [True, False]
}

gs_ridge = gridsearch("Ridge", pipe_ridge, param_grid_ridge, cv=5)


# ============================================================
# 4) EXTRATREES (2 etapas)
# ============================================================
pipe_et = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", ExtraTreesRegressor(random_state=42, n_jobs=-1))
])

param_grid_et_stage1 = {
    "model__n_estimators": [400, 800],
    "model__max_depth": [None, 15, 30],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", 0.7, 1.0]
}
gs_et_1 = gridsearch("ExtraTrees Stage1", pipe_et, param_grid_et_stage1, cv=5)

bp = gs_et_1.best_params_
best_depth = bp["model__max_depth"]
best_leaf  = bp["model__min_samples_leaf"]
best_feat  = bp["model__max_features"]
best_nest  = bp["model__n_estimators"]

depth_candidates = [best_depth]
if best_depth is None:
    depth_candidates += [15, 30]
else:
    depth_candidates += [max(5, best_depth - 5), best_depth + 5]

leaf_candidates = sorted(set([1, best_leaf, max(1, best_leaf - 1), best_leaf + 1, 5]))
feat_candidates = list(dict.fromkeys([best_feat, "sqrt", 0.7, 1.0]))
nest_candidates = sorted(set([max(200, best_nest - 200), best_nest, best_nest + 200, 1200]))

param_grid_et_stage2 = {
    "model__n_estimators": nest_candidates,
    "model__max_depth": depth_candidates,
    "model__min_samples_leaf": leaf_candidates,
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": feat_candidates
}
gs_et_2 = gridsearch("ExtraTrees Stage2", pipe_et, param_grid_et_stage2, cv=5)


# ============================================================
# 5) HistGradientBoosting
# ============================================================
pipe_hgb = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

param_grid_hgb = {
    "model__learning_rate": [0.03, 0.06, 0.1],
    "model__max_depth": [None, 6, 10],
    "model__max_iter": [300, 600],
    "model__min_samples_leaf": [10, 20, 50],
    "model__l2_regularization": [0.0, 0.1, 1.0]
}

gs_hgb = gridsearch("HistGradientBoosting", pipe_hgb, param_grid_hgb, cv=5)


# ============================================================
# 6) DecisionTreeRegressor (exigido)
# ============================================================
pipe_dt = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", DecisionTreeRegressor(random_state=42))
])

param_grid_dt = {
    "model__max_depth": [3, 5, 7, 10, 15, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

gs_dt = gridsearch("DecisionTreeRegressor", pipe_dt, param_grid_dt, cv=5)


# ============================================================
# 7) KNeighborsRegressor (exigido) — dense
# ============================================================
pipe_knn = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", KNeighborsRegressor())
])

param_grid_knn = {
    "model__n_neighbors": [5, 10, 20, 40, 60],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

gs_knn = gridsearch("KNeighborsRegressor", pipe_knn, param_grid_knn, cv=5)


# ============================================================
# 8) SVR (exigido no lugar do SVC) — dense
# ============================================================
pipe_svr = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", SVR(kernel="rbf"))
])

param_grid_svr = {
    "model__C": [10, 100, 300],
    "model__epsilon": [0.05, 0.1, 0.2],
    "model__gamma": ["scale", 0.1]
}

gs_svr = gridsearch("SVR (RBF)", pipe_svr, param_grid_svr, cv=5)


Fitting 5 folds for each of 14 candidates, totalling 70 fits

Ridge best params: {'model__alpha': 300, 'model__fit_intercept': True}
Ridge best CV RMSE(log): 1.4475951029711485
Fitting 5 folds for each of 54 candidates, totalling 270 fits

ExtraTrees Stage1 best params: {'model__max_depth': 30, 'model__max_features': 0.7, 'model__min_samples_leaf': 1, 'model__n_estimators': 800}
ExtraTrees Stage1 best CV RMSE(log): 0.2885808685491401
Fitting 5 folds for each of 324 candidates, totalling 1620 fits

ExtraTrees Stage2 best params: {'model__max_depth': 25, 'model__max_features': 0.7, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 1200}
ExtraTrees Stage2 best CV RMSE(log): 0.28802925379362787
Fitting 5 folds for each of 162 candidates, totalling 810 fits

HistGradientBoosting best params: {'model__l2_regularization': 0.1, 'model__learning_rate': 0.06, 'model__max_depth': None, 'model__max_iter': 300, 'model__min_samples_leaf': 10}
HistGradientBoosting be

In [7]:
# ============================================================
# 9) AVALIAÇÃO FINAL: TREINO + TESTE
# ============================================================
results = []
results.append(eval_train_test(gs_dt.best_estimator_, "DecisionTreeRegressor"))
results.append(eval_train_test(gs_knn.best_estimator_, "KNeighborsRegressor"))
results.append(eval_train_test(gs_svr.best_estimator_, "SVR (RBF)"))

# (opcional) seus modelos anteriores também:
results.append(eval_train_test(gs_ridge.best_estimator_, "Ridge"))
results.append(eval_train_test(gs_et_2.best_estimator_, "ExtraTrees (Stage2 Best)"))
results.append(eval_train_test(gs_hgb.best_estimator_, "HistGradientBoosting"))

res_df = pd.DataFrame(results).sort_values("MAE_R$_teste", ascending=True)
print("\n=== RESUMO (ordenado por MAE_R$ no teste) ===")
print(res_df)



=== DecisionTreeRegressor (TREINO) ===
MAE_R$: 336099.7206597193
RMSE_R$: 6175286.719955112
R2_R$: 0.5413532260744784
MAPE_R$: 0.11316363597224952
MAPE_R$_>=100000: 0.10924913788783613
MAE_log: 0.11171784351791932
RMSE_log: 0.1782852448825589

=== DecisionTreeRegressor (TESTE) ===
MAE_R$: 454563.31143460603
RMSE_R$: 1784824.0314867066
R2_R$: 0.7576038396973521
MAPE_R$: 0.27893372097597674
MAPE_R$_>=100000: 0.271383829854236
MAE_log: 0.24236650493833187
RMSE_log: 0.3922234238819028

=== KNeighborsRegressor (TREINO) ===
MAE_R$: 9.197093986892088e-10
RMSE_R$: 1.4161432477292596e-08
R2_R$: 1.0
MAPE_R$: 4.524689312543221e-16
MAPE_R$_>=100000: 4.583270341080269e-16
MAE_log: 0.0
RMSE_log: 0.0

=== KNeighborsRegressor (TESTE) ===
MAE_R$: 432493.33588306815
RMSE_R$: 1409109.1835800263
R2_R$: 0.8489140021838282
MAPE_R$: 0.30734989310436356
MAPE_R$_>=100000: 0.2893092876184685
MAE_log: 0.32844160843006365
RMSE_log: 0.5355143687954154

=== SVR (RBF) (TREINO) ===
MAE_R$: 61736.74413090266
RMSE_R$:

In [ ]:
import numpy as np
import pandas as pd

# 1) Índices do conjunto de teste (preservados pelo train_test_split)
idx_test = y_test.index

# 2) Pegar as linhas originais do dataframe (com todas as colunas que você quiser mostrar)
# Se "tabela" é seu dataframe original (antes de dropar colunas), use ele:
df_test = tabela.loc[idx_test].copy()

# 3) Previsões (seus modelos treinados em log)
pred_log_ridge = gs_ridge.best_estimator_.predict(X_test)
pred_log_et    = gs_et_2.best_estimator_.predict(X_test)
pred_log_hgb   = gs_hgb.best_estimator_.predict(X_test)

df_test["PRED_RIDGE"]      = np.expm1(pred_log_ridge)
df_test["PRED_EXTRATREES"] = np.expm1(pred_log_et)
df_test["PRED_HISTGB"]     = np.expm1(pred_log_hgb)

# 4) Erros absolutos e percentuais (comparando com o valor real)
df_test["ERRO_ABS_RIDGE"]      = (df_test["PRED_RIDGE"] - df_test["VALOR_VENDA"]).abs()
df_test["ERRO_ABS_EXTRATREES"] = (df_test["PRED_EXTRATREES"] - df_test["VALOR_VENDA"]).abs()
df_test["ERRO_ABS_HISTGB"]     = (df_test["PRED_HISTGB"] - df_test["VALOR_VENDA"]).abs()

df_test["ERRO_PCT_RIDGE"]      = df_test["ERRO_ABS_RIDGE"] / df_test["VALOR_VENDA"]
df_test["ERRO_PCT_EXTRATREES"] = df_test["ERRO_ABS_EXTRATREES"] / df_test["VALOR_VENDA"]
df_test["ERRO_PCT_HISTGB"]     = df_test["ERRO_ABS_HISTGB"] / df_test["VALOR_VENDA"]

# 5) (Opcional) Selecionar só colunas de "características" + resultados
cols_info = [
    "DS_CIDADE", "DS_SETOR", "COD_DESTINACAO_IMOVEL", "ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL",
    "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "PERCENTUAL_ENTRADA", "SITUACAO_VISTORIA", "VALOR_LAUDO",
    "VALOR_VENDA"
]
cols_info = [c for c in cols_info if c in df_test.columns]

cols_result = [
    "PRED_RIDGE", "PRED_EXTRATREES", "PRED_HISTGB",
    "ERRO_ABS_RIDGE", "ERRO_ABS_EXTRATREES", "ERRO_ABS_HISTGB",
    "ERRO_PCT_RIDGE", "ERRO_PCT_EXTRATREES", "ERRO_PCT_HISTGB"
]

df_vis = df_test[cols_info + cols_result].copy()

def moeda_br(v):
    if pd.isna(v):
        return ""
    s = f"{float(v):,.2f}"
    s = s.replace(",", "X").replace(".", ",").replace("X", ".")
    return f"R$ {s}"

def pct_br(v):
    if pd.isna(v):
        return ""
    return f"{float(v)*100:.1f}%".replace(".", ",")

cols_area = ["AREA_MAX_CONSTR", "AREA_BASE", "AREA"]
cols_area = [c for c in cols_area if c in df_vis.columns]

fmt_area = {c: "{:,.2f}".format for c in cols_area}

cols_moeda = ["VALOR_LAUDO", "VALOR_VENDA", "PRED_RIDGE", "PRED_EXTRATREES", "PRED_HISTGB",
             "ERRO_ABS_RIDGE", "ERRO_ABS_EXTRATREES", "ERRO_ABS_HISTGB"]
cols_pct = ["ERRO_PCT_RIDGE", "ERRO_PCT_EXTRATREES", "ERRO_PCT_HISTGB"]

fmt = {}
for c in cols_moeda:
    if c in df_vis.columns:
        fmt[c] = moeda_br
for c in cols_pct:
    if c in df_vis.columns:
        fmt[c] = pct_br

fmt_total = {}
fmt_total.update(fmt)        # fmt que você já tinha (moeda e %)
fmt_total.update(fmt_area)   # adiciona áreas com 2 casas

# exemplo: ordenar pelos maiores valores reais
display(df_vis.sort_values("VALOR_VENDA", ascending=False).head(30).style.format(fmt_total))

,DS_CIDADE,DS_SETOR,COD_DESTINACAO_IMOVEL,ANO_VENDA,NR_EDITAL,ITEM_EDITAL,AREA_MAX_CONSTR,AREA_BASE,AREA,PERCENTUAL_ENTRADA,SITUACAO_VISTORIA,VALOR_LAUDO,VALOR_VENDA,PRED_RIDGE,PRED_EXTRATREES,PRED_HISTGB,ERRO_ABS_RIDGE,ERRO_ABS_EXTRATREES,ERRO_ABS_HISTGB,ERRO_PCT_RIDGE,ERRO_PCT_EXTRATREES,ERRO_PCT_HISTGB
869,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2021,2021-13,5,"6,300.00","6,300.00","1,000.00",30,VAGO,"R$ 19.000.000,00","R$ 31.652.000,00","R$ 10.122.171,56","R$ 29.085.330,09","R$ 30.213.573,83","R$ 21.529.828,44","R$ 2.566.669,91","R$ 1.438.426,17","68,0%","8,1%","4,5%"
1564,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2023,2023-9,2,"6,300.00","6,300.00","1,000.00",30,VAGO,"R$ 19.600.000,00","R$ 31.000.000,00","R$ 11.104.788,02","R$ 32.395.755,42","R$ 26.057.679,36","R$ 19.895.211,98","R$ 1.395.755,42","R$ 4.942.320,64","64,2%","4,5%","15,9%"
439,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2021,2021-1,7,"5,796.00","5,796.00",920.00,5,OBSTRUIDO,"R$ 17.300.000,00","R$ 25.525.000,00","R$ 1.192.209,14","R$ 22.465.974,45","R$ 15.718.148,90","R$ 24.332.790,86","R$ 3.059.025,55","R$ 9.806.851,10","95,3%","12,0%","38,4%"
862,SANTA MARIA,POLO DESENV/ECONOMICO JUSCELINO KUBITSCHEK INDUSTRIA COMERCIO DE APOIO,60225,2021,2021-12,91,"181,131.60","75,471.50","75,471.50",5,VAGO,"R$ 23.700.000,00","R$ 24.000.003,50","R$ 602.404.283,41","R$ 10.696.460,79","R$ 10.179.636,85","R$ 578.404.279,91","R$ 13.303.542,71","R$ 13.820.366,65","2410,0%","55,4%","57,6%"
442,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2021,2021-1,10,"5,796.00","5,796.00",920.00,5,OBSTRUIDO,"R$ 17.300.000,00","R$ 23.888.000,00","R$ 1.154.348,51","R$ 22.514.020,40","R$ 15.156.142,90","R$ 22.733.651,49","R$ 1.373.979,60","R$ 8.731.857,10","95,2%","5,8%","36,6%"
233,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2020,2020-11,13,"5,796.00","5,796.00",920.00,30,VAGO,"R$ 14.900.000,00","R$ 22.812.477,00","R$ 9.633.906,32","R$ 20.922.716,21","R$ 21.992.783,58","R$ 13.178.570,68","R$ 1.889.760,79","R$ 819.693,42","57,8%","8,3%","3,6%"
691,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2021,2021-7,2,"5,796.00","5,796.00",920.00,30,VAGO,"R$ 17.300.000,00","R$ 22.612.000,00","R$ 10.348.898,65","R$ 27.891.158,00","R$ 22.223.380,65","R$ 12.263.101,35","R$ 5.279.158,00","R$ 388.619,35","54,2%","23,3%","1,7%"
228,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2020,2020-11,8,"5,796.00","5,796.00",920.00,30,VAGO,"R$ 14.900.000,00","R$ 21.801.018,00","R$ 9.837.070,09","R$ 20.907.570,16","R$ 22.781.935,70","R$ 11.963.947,91","R$ 893.447,84","R$ 980.917,70","54,9%","4,1%","4,5%"
235,BRASILIA,SHCNW-SETOR DE HABITAÇÕES COLETIVAS NOROESTE,1893,2020,2020-11,15,"5,796.00","5,796.00",920.00,30,OBSTRUIDO,"R$ 14.900.000,00","R$ 21.604.012,00","R$ 10.436.419,28","R$ 20.937.226,24","R$ 24.651.946,53","R$ 11.167.592,72","R$ 666.785,76","R$ 3.047.934,53","51,7%","3,1%","14,1%"
1851,BRASILIA,SETOR DE EMBAIXADAS NORTE,60308,2024,2024-2,12,"9,551.47","9,551.47","7,959.56",5,VAGO,"R$ 15.800.000,00","R$ 20.888.000,00","R$ 1.321.720,29","R$ 14.140.910,20","R$ 9.340.717,53","R$ 19.566.279,71","R$ 6.747.089,80","R$ 11.547.282,47","93,7%","32,3%","55,3%"


In [10]:
print("Máximo na base inteira:", tabela["VALOR_VENDA"].max())
print("Máximo no teste:", y_test.max())
print("Máximo no treino:", y_train.max())

# ver em qual conjunto está o outlier
idx_max = tabela["VALOR_VENDA"].idxmax()
print("Índice do maior valor:", idx_max)
print("Maior valor:", tabela.loc[idx_max, "VALOR_VENDA"])

print("Está no teste?", idx_max in y_test.index)
print("Está no treino?", idx_max in y_train.index)

Máximo na base inteira: 406670000.0
Máximo no teste: 31652000.0
Máximo no treino: 406670000.0
Índice do maior valor: 879
Maior valor: 406670000.0
Está no teste? False
Está no treino? True


In [11]:
print("Top 10 maiores VALOR_VENDA no teste:")
print(y_test.sort_values(ascending=False).head(10))

print("\nTop 10 menores VALOR_VENDA no teste:")
print(y_test.sort_values().head(10))

Top 10 maiores VALOR_VENDA no teste:
869     31652000.0
1564    31000000.0
439     25525000.0
862     24000003.5
442     23888000.0
233     22812477.0
691     22612000.0
228     21801018.0
235     21604012.0
1851    20888000.0
Name: VALOR_VENDA, dtype: float64

Top 10 menores VALOR_VENDA no teste:
1005    36000.0
830     36155.0
1385    40500.0
538     41106.0
136     42010.0
135     42010.0
371     42200.0
574     44000.0
2921    45700.0
406     46800.0
Name: VALOR_VENDA, dtype: float64


In [12]:
num_check = ["AREA_MAX_CONSTR","AREA_BASE","AREA","PERCENTUAL_ENTRADA"]
print(X_train[num_check].describe().T)

                     count         mean          std   min     25%      50%  \
AREA_MAX_CONSTR     2258.0  1602.281711  7764.473104  4.00  315.00  472.385   
AREA_BASE           2258.0  1163.476162  7263.981017  4.00  200.00  420.000   
AREA                2258.0   698.423589  4016.803129  4.00  105.00  292.340   
PERCENTUAL_ENTRADA  2258.0     0.054074     0.029972  0.05    0.05    0.050   

                        75%       max  
AREA_MAX_CONSTR     1305.00  328168.0  
AREA_BASE           1035.00  328168.0  
AREA                 700.00  164084.0  
PERCENTUAL_ENTRADA     0.05       0.3  


In [14]:
pred_log_ridge = gs_ridge.best_estimator_.predict(X_test)
pred_ridge = np.expm1(pred_log_ridge)

df_diag = pd.DataFrame({
    "REAL": y_test.values,
    "PRED_RIDGE": pred_ridge,
    "ERRO_ABS": np.abs(pred_ridge - y_test.values),
    "ERRO_PCT": np.abs(pred_ridge - y_test.values) / y_test.values
}, index=y_test.index)

print("Top 10 maiores previsões do Ridge:")
display(df_diag.sort_values("PRED_RIDGE", ascending=False).head(10))

print("Top 10 maiores erros absolutos do Ridge:")
display(df_diag.sort_values("ERRO_ABS", ascending=False).head(10))

Top 10 maiores previsões do Ridge:


,REAL,PRED_RIDGE,ERRO_ABS,ERRO_PCT
2571,6659999.0,2.512214e+09,2.505554e+09,376.209300
862,24000003.5,6.024043e+08,5.784043e+08,24.100175
1564,31000000.0,1.110479e+07,1.989521e+07,0.641781
235,21604012.0,1.043642e+07,1.116759e+07,0.516922
691,22612000.0,1.034890e+07,1.226310e+07,0.542327
869,31652000.0,1.012217e+07,2.152983e+07,0.680204
228,21801018.0,9.837070e+06,1.196395e+07,0.548779
233,22812477.0,9.633906e+06,1.317857e+07,0.577691
198,1208000.0,7.946278e+06,6.738278e+06,5.578045
765,18101000.0,6.734037e+06,1.136696e+07,0.627974


Top 10 maiores erros absolutos do Ridge:


,REAL,PRED_RIDGE,ERRO_ABS,ERRO_PCT
2571,6659999.0,2.512214e+09,2.505554e+09,376.209300
862,24000003.5,6.024043e+08,5.784043e+08,24.100175
439,25525000.0,1.192209e+06,2.433279e+07,0.953292
442,23888000.0,1.154349e+06,2.273365e+07,0.951677
869,31652000.0,1.012217e+07,2.152983e+07,0.680204
1564,31000000.0,1.110479e+07,1.989521e+07,0.641781
1851,20888000.0,1.321720e+06,1.956628e+07,0.936723
233,22812477.0,9.633906e+06,1.317857e+07,0.577691
190,14109999.0,1.032363e+06,1.307764e+07,0.926835
691,22612000.0,1.034890e+07,1.226310e+07,0.542327
